# GNEISS: line of sight through electron density

Inspect the **VEE → rocket 397** ray at **2026-02-10 10:24:00 UTC**. The left panel samples the radial plane containing both endpoints. Ground distance is surface arc length, so a straight Cartesian ray can appear curved on this altitude plot. The right panel shows local density and accumulated **modeled** TEC.

Gray indicates unsupported model coverage, never zero density. Cumulative TEC includes only segments with two supported endpoints, exactly as in the calculator. The spherical geographic convention is retained; this view does not independently validate the GPS datum or grid mapping.

Use the shared `lab317/.venv` Python kernel. Run the cells in order. The final cell provides a time slider and pair selectors.

**Smallgrid time selection:** all timestamped output files are included at their original UTC times. No offset, blank-frame exclusion, or corrected-time manifest from the previous run is applied. This run spans 09:50–10:30 UTC at one-minute cadence. Density volumes are available throughout; ray plots require trajectory coverage. Restart the kernel and Run All after switching runs.

In [ ]:
from pathlib import Path
import sys
print(sys.executable)

ROOT = Path.cwd()
if not (ROOT / "src/gemini_density_tec").exists():
    ROOT = ROOT.parent
if not (ROOT / "src/gemini_density_tec").exists():
    raise RuntimeError("Open this notebook from the gemini_density_tec repository")
sys.path.insert(0, str(ROOT / "src"))
import matplotlib.pyplot as plt
from gemini_density_tec.los import LOSViewer

MODEL = ROOT / "data/gneiss_20260210_35400_smallgrid"
# Reuse the normalized measured rocket/receiver inputs, not the old time manifest.
GEOMETRY_INPUTS = ROOT / "data/gneiss_20260210_35400_09_SD_UM_xa_gneiss"
viewer = LOSViewer(MODEL, time_source="filename",
                   trajectories=GEOMETRY_INPUTS / "trajectories.csv",
                   receivers=GEOMETRY_INPUTS / "receivers.csv")
EXPORT_DIR = ROOT / "results" / MODEL.name
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f"{len(viewer.frames)} original frame times available")

## Select a frame and render/export

The default frame is 10:24:00 UTC. Density limits stay fixed at 10⁸–2×10¹² m⁻³; adjust them below if needed. One full density frame is retained in memory, and changing frames replaces it. Each redraw samples a 400×350 slice.

In [ ]:
RECEIVER = "VEE"
ROCKET = "397"
FRAME_INDEX = next(i for i, r in enumerate(viewer.frames) if r["model_time_utc"] == "2026-02-10T10:24:00+00:00")
STEP_M = 500
DENSITY_LIMITS = (1e8, 2e12)

fig, diagnostic = viewer.plot(
    FRAME_INDEX, RECEIVER, ROCKET, step_m=STEP_M,
    density_limits=DENSITY_LIMITS,
    export=EXPORT_DIR / "vee_397_los_102400.png",
)
plt.show()
print({key: diagnostic[key] for key in
       ["model_time_utc", "source_filename", "modeled_stec_tecu", "coverage_fraction", "status"]})

## Inspect the samples

These are the exact 500 m-or-finer samples used for the TEC integration. `cumulative_modeled_tecu` sums supported intervals only; a plateau in an excluded interval does not imply zero physical electron content.

In [ ]:
import pandas as pd
samples = pd.DataFrame({
    "distance_km": diagnostic["distance_m"] / 1000,
    "electron_density_m3": diagnostic["electron_density_m3"],
    "cumulative_modeled_tecu": diagnostic["cumulative_modeled_tecu"],
})
samples.to_csv(EXPORT_DIR / "vee_397_los_102400_samples.csv", index=False)
samples.iloc[::max(1, len(samples)//12)]

## Time slider and pair selection

Release the time slider to load a frame. The controls below do not overwrite the exported default figure or sample CSV. Change the settings in the earlier cell and its export filename to save another view.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

slider = widgets.SelectionSlider(
    options=[(r["model_time_utc"][11:19], i) for i, r in enumerate(viewer.frames)],
    value=FRAME_INDEX, description="UTC", continuous_update=False,
    layout=widgets.Layout(width="550px"),
)
receiver = widgets.Dropdown(options=sorted(viewer.receivers), value=RECEIVER, description="Receiver")
rocket = widgets.Dropdown(options=sorted(viewer.tracks), value=ROCKET, description="Rocket")
def redraw(index, receiver_id, rocket_id):
    figure, _ = viewer.plot(index, receiver_id, rocket_id,
                           step_m=STEP_M, density_limits=DENSITY_LIMITS)
    display(figure)
    plt.close(figure)
output = widgets.interactive_output(redraw, {
    "index": slider, "receiver_id": receiver, "rocket_id": rocket,
})
display(widgets.VBox([slider, widgets.HBox([receiver, rocket]), output]))

## Density volume and timestamp audit — geodetic coordinates

Run the setup cell first. Choose the frame, WGS84 ellipsoidal height (km), longitude (degrees east; negative in Alaska), and geodetic latitude (degrees north), then click **Inspect volume**.

The panels show longitude–latitude at fixed height, longitude–height at fixed latitude, and latitude–height at fixed longitude. Density is resampled onto 250×250 grids: WGS84 coordinates → Earth-centered XYZ → GEMINI spherical coordinates → native model interpolation. This is a coordinate transformation, not a relabeling of magnetic axes. Gray marks locations outside the model. Colors stay fixed across frames.

This changes only volume visualization; the existing rocket-ray calculation still uses its documented spherical input convention. WGS84 heights are relative to the ellipsoid, not sea level.

The audit compares filename UTC, embedded HDF5 UTC, and the model UTC used in plots. For smallgrid, model UTC equals filename UTC without correction.

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from datetime import datetime, timedelta, timezone
import ipywidgets as widgets
from IPython.display import display, clear_output
from gemini_density_tec.cli import utc

# Audit all files without reading their density volumes.
audit_records = []
for index, record in enumerate(viewer.frames):
    filename = record["filename"]
    day, seconds = Path(filename).stem.split("_", 1)
    filename_time = datetime.strptime(day, "%Y%m%d").replace(tzinfo=timezone.utc) + timedelta(seconds=float(seconds))
    path = MODEL / filename
    embedded_time = None
    if path.exists():
        with h5py.File(path, "r") as handle:
            if "time/ymd" in handle and "time/UThour" in handle:
                ymd = np.asarray(handle["time/ymd"][()]).ravel()
                hour = float(np.asarray(handle["time/UThour"][()]).item())
                embedded_time = datetime(*map(int, ymd), tzinfo=timezone.utc) + timedelta(hours=hour)
    corrected = utc(record["model_time_utc"])
    audit_records.append({
        "index": index, "filename": filename, "available": path.exists(),
        "filename_utc": filename_time, "embedded_utc": embedded_time,
        "model_utc": corrected,
        "model_minus_filename_s": (corrected-filename_time).total_seconds(),
        "embedded_minus_filename_s": (embedded_time-filename_time).total_seconds() if embedded_time else np.nan,
    })
time_audit = pd.DataFrame(audit_records)
display(time_audit)

from pyproj import Transformer
from gemini3d.grid.convert import Re, geomag2geog
from gemini_density_tec.tec import xyz, geographic

# Actual WGS84 geodetic coordinates, transformed through Earth-centered XYZ
# to GEMINI spherical coordinates before native-grid density interpolation.
to_ecef = Transformer.from_crs("EPSG:4979", "EPSG:4978", always_xy=True)
to_geodetic = Transformer.from_crs("EPSG:4978", "EPSG:4979", always_xy=True)
if viewer.model.dipole:
    raise ValueError("Automatic plotting bounds currently support this Cartesian run only.")
model = viewer.model
az, ax2, ax3 = [np.linspace(a.min(), a.max(), count) for a, count in zip(model.axes, (8, 40, 40))]
zz, xx, yy = np.meshgrid(az, ax2, ax3, indexing="ij")
phi = model.phi0 + xx.ravel()/(Re*np.sin(model.theta0))
theta = model.theta0 - yy.ravel()/Re
spherical_lon, spherical_lat = geomag2geog(phi, theta)
physical_xyz = xyz(spherical_lat, spherical_lon, zz.ravel())
glon, glat, height = to_geodetic.transform(*physical_xyz.T)
longitude_axis = np.linspace(np.min(glon), np.max(glon), 250)
latitude_axis = np.linspace(np.min(glat), np.max(glat), 250)
height_axis = np.linspace(np.min(height)/1000, np.max(height)/1000, 250)

def inspect_volume(frame_index, altitude_km, longitude_deg, latitude_deg):
    record = viewer.frames[frame_index]
    path = MODEL / record["filename"]
    if viewer._frame != path:
        viewer._density = None
        viewer._density = viewer.model.density(path)
        viewer._frame = path
    def sample_geodetic(lon, lat, height_km):
        lon, lat, h = np.broadcast_arrays(lon, lat, np.asarray(height_km)*1000)
        x, y, z = to_ecef.transform(lon.ravel(), lat.ravel(), h.ravel())
        a, o, l = geographic(np.column_stack((x, y, z)))
        return viewer._density(model.coordinates(a, o, l)).reshape(lon.shape)
    lon_map, lat_map = np.meshgrid(longitude_axis, latitude_axis)
    lon_section, h_lon = np.meshgrid(longitude_axis, height_axis)
    lat_section, h_lat = np.meshgrid(latitude_axis, height_axis)
    fig, plots = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)
    norm = LogNorm(1e8, 2e12)
    views = [
        (longitude_axis, latitude_axis, sample_geodetic(lon_map, lat_map, altitude_km),
         "Longitude [°E]", "Geodetic latitude [°N]", f"WGS84 height = {altitude_km:.1f} km", longitude_deg, latitude_deg),
        (longitude_axis, height_axis, sample_geodetic(lon_section, latitude_deg, h_lon),
         "Longitude [°E]", "WGS84 ellipsoidal height [km]", f"Latitude = {latitude_deg:.2f}°N", longitude_deg, altitude_km),
        (latitude_axis, height_axis, sample_geodetic(longitude_deg, lat_section, h_lat),
         "Geodetic latitude [°N]", "WGS84 ellipsoidal height [km]", f"Longitude = {longitude_deg:.2f}°E", latitude_deg, altitude_km),
    ]
    for plot, (horizontal, vertical, values, xlabel, ylabel, title, cross_x, cross_y) in zip(plots, views):
        plot.set_facecolor("#d6d6d6")
        color = plot.pcolormesh(horizontal, vertical, np.ma.masked_invalid(values),
                                shading="auto", cmap="viridis", norm=norm, rasterized=True)
        plot.axvline(cross_x, color="white", linestyle="--", linewidth=.8)
        plot.axhline(cross_y, color="white", linestyle="--", linewidth=.8)
        plot.set(xlabel=xlabel, ylabel=ylabel, title=title)
        plot.ticklabel_format(useOffset=False)
    fig.colorbar(color, ax=list(plots), label="Electron density [m⁻³]", extend="both")
    row = time_audit.iloc[frame_index]
    fig.suptitle(f"{record['filename']}\nFile UTC: {row['filename_utc']} | Model UTC: {row['model_utc']}\nWGS84 geodetic slices • gray = outside model", fontsize=11)
    display(fig)
    plt.close(fig)
    display(time_audit.iloc[[frame_index]])
    return fig

volume_frame = widgets.Dropdown(
    options=[(f"{i}: {r['filename']} → UTC {r['model_time_utc'][11:19]}", i) for i, r in enumerate(viewer.frames)],
    value=FRAME_INDEX, description="Frame", layout=widgets.Layout(width="750px"))
def slice_slider(axis, value, label, step=1):
    return widgets.FloatSlider(min=float(axis.min()), max=float(axis.max()),
                               value=float(np.clip(value, axis.min(), axis.max())),
                               step=step, description=label, continuous_update=False)
volume_z = slice_slider(height_axis, 110, "Height km")
volume_x = slice_slider(longitude_axis, float(np.median(glon)), "Longitude", .01)
volume_y = slice_slider(latitude_axis, float(np.median(glat)), "Latitude", .01)
volume_button = widgets.Button(description="Inspect volume", button_style="primary")
volume_output = widgets.Output()
def redraw_volume(_=None):
    with volume_output:
        clear_output(wait=True)
        inspect_volume(volume_frame.value, volume_z.value, volume_x.value, volume_y.value)
volume_button.on_click(redraw_volume)
display(widgets.VBox([volume_frame, widgets.HBox([volume_z, volume_x, volume_y]), volume_button, volume_output]))
redraw_volume()
